# Split the data prior to normalization. This is to avoid data leakage from the test set into the training set during normalization.

## Splitting strategy 
Splits are generated per well_fov. The index of the split is the well_fov. The split is generated by randomly sampling 80% of the cells in the well_fov for training and 20% for testing. The random seed is set to 0 for reproducibility.

## Import libraries

In [1]:
import argparse
import os
import pathlib

import pandas as pd
from sklearn.model_selection import train_test_split
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

## Set paths and variables

In [2]:
if in_notebook:
    import tqdm.notebook as tqdm

    plate_name = "plate_2"
else:
    import tqdm

    argparser = argparse.ArgumentParser()

    argparser.add_argument(
        "--plate_name",
        type=str,
        help="Name of the plate to analyze",
    )
    args = argparser.parse_args()
    plate_name = args.plate_name

In [3]:
TEST_SPLIT_SEED = 0
TRAIN_SPLIT_FRAC = 0.8


def create_train_test_split(
    df: pd.DataFrame, train_frac: float = 0.8, seed: int = 0
) -> pd.DataFrame:
    """
    Create the combined train_test_index split dataframe

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing image-based profile per well_fov

    train_frac : float, optional
        Fraction of the data to use for training, by default 0.8
    seed : int, optional
        Random seed for reproducibility, by default 0

    Returns
    -------
    pd.DataFrame
        The combined train_test_index split dataframe
    """

    # get the train and test splits.
    df.reset_index(inplace=True)
    train, test = train_test_split(
        df,  # no need to stratify by well_fov since we are splitting by well_fov already
        train_size=train_frac,
        random_state=seed,
        shuffle=True,
    )
    # retrieve the index of the train and test splits
    train_index = train["index"].values
    test_index = test["index"].values
    # save the train and test splits as parquet files
    train_index_df = pd.DataFrame({"index": train_index})
    train_index_df.insert(0, "train_test_split", "train")
    test_index_df = pd.DataFrame({"index": test_index})
    test_index_df.insert(0, "train_test_split", "test")
    train_test_split_df = pd.concat([train_index_df, test_index_df], axis=0)
    return train_test_split_df

In [4]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve()

image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(f"{image_base_dir}/processed_data/").resolve(strict=True)
sc_tracks_path = pathlib.Path(
    f"{image_base_dir}/9.single_Cell_tracks_merged/{plate_name}/"
).resolve(strict=True)
sc_track_profiles_list = list(sc_tracks_path.glob("*.parquet"))
train_test_split_path = pathlib.Path(
    f"{root_dir}/Wave2_data/6.image_based_profiling/train_test_split_indices/{plate_name}/"
).resolve()
train_test_split_path.mkdir(exist_ok=True, parents=True)

In [5]:
for well_fov_profile_path in tqdm.tqdm(
    sc_track_profiles_list, total=len(sc_track_profiles_list)
):
    well_fov = well_fov_profile_path.stem.replace("_sc_tracks_merged", "")

    train_test_split_output_file_path = (
        train_test_split_path / f"{well_fov}_train_test_split.parquet"
    )
    if train_test_split_output_file_path.exists():
        continue
    df = pd.read_parquet(well_fov_profile_path)
    train_test_split_df = create_train_test_split(
        df=df,
        train_frac=TRAIN_SPLIT_FRAC,
        seed=TEST_SPLIT_SEED,
    )
    train_test_split_df.insert(0, "well_fov", well_fov)
    train_test_split_df.to_parquet(train_test_split_output_file_path, index=False)

  0%|          | 0/2 [00:00<?, ?it/s]